# Learning a price differs from learning a hedge
Pricing maps the current state to one value. Hedging repeatedly changes an action as the state evolves. This notebook makes both ledgers visible.

## 1. Price and delta with SciPy
Use the standard normal distribution implementation for the Black–Scholes calculation.

In [ ]:
import numpy as np
from scipy.stats import norm

spot, strike, rate, volatility, maturity = 100, 100, .03, .20, 1.0
d1 = (np.log(spot / strike) + (rate + volatility**2 / 2) * maturity) / (volatility * np.sqrt(maturity))
d2 = d1 - volatility * np.sqrt(maturity)
price = spot * norm.cdf(d1) - strike * np.exp(-rate * maturity) * norm.cdf(d2)
delta = norm.cdf(d1)
print(f'price={price:.2f}, initial delta={delta:.3f}')

## 2. Build a discrete hedge from scratch
Implement the normal CDF, simulate one price path, accrue cash, and rebalance the shares each week.

In [ ]:
import math, random

normal_cdf = lambda x: .5 * (1 + math.erf(x / math.sqrt(2)))
steps, dt = 52, 1 / 52
random.seed(9)
scratch_spot = 100.0
d1 = (math.log(scratch_spot / strike) + (rate + volatility**2 / 2)) / volatility
d2 = d1 - volatility
scratch_price = scratch_spot * normal_cdf(d1) - strike * math.exp(-rate) * normal_cdf(d2)
shares = normal_cdf(d1)
cash = scratch_price - shares * scratch_spot

for step in range(steps):
    cash *= math.exp(rate * dt)
    scratch_spot *= math.exp((rate - volatility**2 / 2) * dt + volatility * math.sqrt(dt) * random.gauss(0, 1))
    if step < steps - 1:
        time_left = 1 - (step + 1) * dt
        d1 = (math.log(scratch_spot / strike) + (rate + volatility**2 / 2) * time_left) / (volatility * math.sqrt(time_left))
        new_shares = normal_cdf(d1)
        cash -= (new_shares - shares) * scratch_spot
        shares = new_shares
payoff = max(scratch_spot - strike, 0)
print(f'ending spot={scratch_spot:.2f}, hedge error={cash + shares * scratch_spot - payoff:.2f}')

## What to notice
The price is one calculation at time zero. The hedge is a path of holdings and cash. Discrete rebalancing leaves an error even in the same model world.